# TetraFT — Phase 2 scout (0.8B on Kaggle)

**Attach datasets**
- `tetraft-code` — flat `.py` + this notebook (refresh after code changes)
- `tetraft-fineweb-edu-50m` — `train.jsonl`, `val.jsonl`

| Setting | Value |
|---------|--------|
| Accelerator | **GPU** |
| Internet | **ON** first run (Qwen + recent `transformers` for `qwen3_5`) |
| Flow | inventory → original PPL → shock PPL → QAFT |

**Control DNA:** `full_smoke` (λ_warmup=256, peak lr 2e-4, linear→0, c=0.25, absmean_channel).

| Preset | Steps | ≈ tokens | Notes |
|--------|------:|---------:|-------|
| `short` | 200 | 0.82M | pipeline only |
| **`full_smoke`** | 1280 | **5.24M** | **default scout**; frozen control PPL ~79.4 |
| `scale_25m` | 6104 | 25M | recorded partial (~69); **not** default |
| `scale_50m` | 12207 | 50M | deprioritized |

**Disk-safe defaults (in code):** weights-only `best`/`final`, `save_steps=0` (no step dumps), `metrics.jsonl`. Use `save_optimizer=True` only for resume.

**Phase 2 scope arm:** `SKIP_LINEAR_ATTN=True` leaves Qwen3.5 GDN (`linear_attn.*`) in FP.

Logic in `run_smoke.py` — notebook is glue only. Numbers: `RESULTS.md`.

In [ ]:
# Qwen3.5 needs recent transformers (model_type qwen3_5).
# If KeyError qwen3_5:
# %pip install -U "git+https://github.com/huggingface/transformers.git"
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
import sys
from pathlib import Path

def find_file(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.is_file():
            return direct
        for p in root.rglob(name):
            if p.is_file():
                return p
    raise FileNotFoundError(name)

code_py = find_file("run_smoke.py")
code_root = code_py.parent
sys.path.insert(0, str(code_root))
print("code:", code_root)

train_path = find_file("train.jsonl")
val_path = find_file("val.jsonl")
print("train:", train_path)
print("val:", val_path)

import transformers
print("transformers", transformers.__version__)

In [ ]:
from run_smoke import run_smoke
import argparse
import shutil
from pathlib import Path

# --- Phase 2 scout knobs (one factor at a time) ---
PRESET = "full_smoke"  # control DNA; do not default to scale_25m
SKIP_LINEAR_ATTN = True  # False = control (quantize GDN); True = exclude linear_attn
SAVE_OPTIMIZER = False  # True only if you need resume (large)
CLEAR_OUTPUT = True  # wipe output_dir before run (Kaggle disk hygiene)

OUTPUT_DIR = (
    "/kaggle/working/checkpoints_no_gdn"
    if SKIP_LINEAR_ATTN
    else "/kaggle/working/checkpoints_ctrl"
)

out = Path(OUTPUT_DIR)
if CLEAR_OUTPUT and out.exists():
    shutil.rmtree(out)
    print("cleared", out)
out.mkdir(parents=True, exist_ok=True)

ns = argparse.Namespace(
    preset=PRESET,
    model_name=None,
    train_data=str(train_path),
    val_data=str(val_path),
    output_dir=OUTPUT_DIR,
    seq_length=None,
    batch_size=None,
    max_steps=None,
    max_eval_batches=20,
    max_train_texts=None,
    max_val_texts=None,
    skip_train=False,
    skip_shock=False,
    no_bf16=False,
    no_8bit_adam=False,
    quant_warmup_steps=None,
    warmup_steps=None,
    learning_rate=None,
    lr_scheduler=None,
    min_lr_ratio=None,
    logging_steps=None,
    eval_steps=None,
    save_steps=None,  # 0 via preset/config = no periodic step_* dumps
    save_optimizer=SAVE_OPTIMIZER,
    skip_linear_attn=SKIP_LINEAR_ATTN,
    seed=42,
    device_map="auto",
)
print(
    f"run preset={PRESET} skip_linear_attn={SKIP_LINEAR_ATTN} "
    f"save_optimizer={SAVE_OPTIMIZER} out={OUTPUT_DIR}"
)
results = run_smoke(ns)
keys = [
    "preset", "ppl_original", "ppl_shock", "ppl_after_smoke",
    "loss_finite", "tokens_seen", "tokens_budget", "steps_ran",
]
print({k: results[k] for k in keys if k in results})
if "inventory_summary" in results:
    print("inventory", results["inventory_summary"])
if "ppl_after_smoke" in results and "ppl_original" in results and results["ppl_original"]:
    print("after/orig =", results["ppl_after_smoke"] / results["ppl_original"])
print("compare to frozen control full_smoke PPL ~79.4 (after/orig ~4.5) when SKIP_LINEAR_ATTN=True")

### Artifacts

Under `OUTPUT_DIR` (default `checkpoints_no_gdn` or `checkpoints_ctrl`):

- `linear_inventory.json` — eligible vs skipped (GDN shows `skip_linear_attn` when on)
- `metrics.jsonl` — loss / PPL / λ rows (no full model state)
- `smoke_results.json`
- `checkpoint-best`, `checkpoint-final` — **weights-only** unless `SAVE_OPTIMIZER=True`

### Frozen baselines (see `RESULTS.md`)

| Run | ≈ tokens | Val PPL |
|-----|---------:|--------:|
| Original FP | — | ~17.7 |
| Shock | 0 | ≫1e6 |
| `full_smoke` control | 5.2M | **~79.4** |
| `scale_25m` (partial) | ~21M | ~68.6 |

### After this run

1. Record: preset, `skip_linear_attn`, tokens, end PPL, after/orig, inventory % quantized.
2. Scope win = clearly better than ~79 at same ~5M budget → keep skip for later scouts.
3. Next factors (separate jobs): c, scale_mode — not bundled with scope.
4. Do **not** default to blind `scale_25m` / `scale_50m`.